# Evaluation - Invitation Extraction

This notebook evaluates the performance of the invitation extraction system against a ground truth dataset of 35 test emails.

In [1]:
from typing import Any

import pandas as pd

# Import the test dataset
from emails_dataset import (
    TEST_EMAILS,
)
from fuzzywuzzy import fuzz

from box2.triage.invitation_extraction import extract_invitation
from box2.triage.models import Invitation, NotInvitation, SafeDocument
from box2.triage.models.document import generate_document_id

## Helper functions

In [ ]:
def create_safe_document_from_test(test_case: dict) -> SafeDocument:
    """Convert a test case dictionary to a SafeDocument for extraction."""
    content = f"{test_case['subject']}\n{test_case['body']}"
    doc_id = generate_document_id(content, prefix="email")

    return SafeDocument(
        document_id=doc_id,
        filename=f"{test_case['email_id']}.eml",
        source_type="email",
        safe_text=content,
        document_timestamp=test_case["received_date"],
        pii_extracted={"emails": [], "phone_numbers": []},
        links_extracted=[],
    )


def normalise_date(date_str: Any) -> str | None:
    """Normalise date to YYYY-MM-DD format for comparison."""
    if pd.isna(date_str) or date_str is None:
        return None
    try:
        dt = pd.to_datetime(date_str)
        # If the year is 1, it's a parsing artifact from "March 12th"
        if dt.year == 1:
            dt = dt.replace(year=2026)  # Use current context year
        return dt.strftime("%Y-%m-%d")
    except:
        return None


def normalise_text_list(items: list[str] | None) -> set:
    """Normalise a list of strings for comparison (lowercase, stripped)."""
    if not items:
        return set()
    return {item.lower().strip() for item in items}


def fuzzy_match(expected: str | None, actual: str | None, threshold: int = 85) -> bool:
    """Check if two strings are 'close enough' using Levenshtein distance."""
    if not expected and not actual:
        return True
    if not expected or not actual:
        return False

    # partial_ratio is great for "Royal Society" vs "The Royal Society"
    return fuzz.partial_ratio(str(expected).lower(), str(actual).lower()) >= threshold


def compare_topics(expected: list[str], actual: list[str], threshold: float = 0.5) -> dict[str, Any]:
    """
    Compare topics with fuzzy matching and a success threshold.
    """
    expected_list = [t.lower().strip() for t in expected]
    actual_list = [t.lower().strip() for t in actual]

    if not expected_list and not actual_list:
        return {
            "is_correct": True,
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "missing": [],
            "extra": [],
        }

    # Find matches using fuzzy logic instead of exact set intersection
    matches = []
    for exp in expected_list:
        for act in actual_list:
            if fuzzy_match(exp, act, threshold=80):  # Topic matching can be slightly looser
                matches.append(exp)
                break

    correct_count = len(matches)
    precision = correct_count / len(actual_list) if actual_list else 0.0
    recall = correct_count / len(expected_list) if expected_list else 0.0

    # We define "correct" based on your 50% threshold requirement
    is_correct = recall >= threshold

    return {
        "is_correct": is_correct,
        "precision": precision,
        "recall": recall,
        "missing": list(set(expected_list) - set(matches)),
        "extra": list(set(actual_list) - set(matches)),
    }


def evaluate_single_extraction(test_case: dict, result: Invitation | NotInvitation) -> dict[str, Any]:
    """
    Evaluate a single extraction result against ground truth.
    """
    email_id = test_case["email_id"]
    expected_is_invitation = test_case["is_invitation"]
    is_invitation = isinstance(result, Invitation)

    evaluation = {
        "email_id": email_id,
        "subject": test_case["subject"],
        "expected_is_invitation": expected_is_invitation,
        "actual_is_invitation": is_invitation,
        "classification_correct": expected_is_invitation == is_invitation,
    }

    if not evaluation["classification_correct"]:
        # ... (keep your existing error_type logic) ...
        return evaluation

    if not is_invitation:
        evaluation["error_type"] = None
        return evaluation

    invitation_result = result

    # 1. EVENT TYPE
    expected_event_type = test_case.get("expected_event_type")
    actual_event_type = (
        invitation_result.event_type.value
        if hasattr(invitation_result.event_type, "value")
        else str(invitation_result.event_type)
    )
    evaluation["expected_event_type"] = expected_event_type
    evaluation["actual_event_type"] = actual_event_type
    evaluation["event_type_correct"] = expected_event_type == actual_event_type

    # 2. HOST ORG
    expected_host = test_case.get("expected_host_org")
    actual_host = invitation_result.host_org
    evaluation["expected_host_org"] = expected_host
    evaluation["actual_host_org"] = actual_host
    evaluation["host_org_correct"] = fuzzy_match(expected_host, actual_host) if expected_host else True

    # 3. DATE
    expected_date = normalise_date(test_case.get("expected_date"))
    actual_date = None
    if invitation_result.proposed_times:
        for proposed_time in invitation_result.proposed_times:
            normalised = normalise_date(proposed_time)
            if normalised:
                actual_date = normalised
                break

    evaluation["expected_date"] = expected_date
    evaluation["actual_date"] = actual_date
    evaluation["date_correct"] = (expected_date == actual_date) or (expected_date is None)

    # 4. LOCATION
    expected_location = test_case.get("expected_location")
    actual_location = invitation_result.location
    evaluation["expected_location"] = expected_location
    evaluation["actual_location"] = actual_location
    evaluation["location_correct"] = fuzzy_match(expected_location, actual_location)

    # 5. TOPICS
    expected_topics = test_case.get("expected_topics", [])
    actual_topics = invitation_result.topics or []
    topic_metrics = compare_topics(expected_topics, actual_topics, threshold=0.5)

    evaluation["expected_topics"] = expected_topics
    evaluation["actual_topics"] = actual_topics
    evaluation["topics_correct"] = topic_metrics["is_correct"]
    evaluation["topics_precision"] = topic_metrics["precision"]
    evaluation["topics_recall"] = topic_metrics["recall"]
    evaluation["topics_missing"] = topic_metrics["missing"]
    evaluation["topics_extra"] = topic_metrics["extra"]

    # 6. OVERALL ACCURACY
    fields_correct = [
        evaluation.get("event_type_correct", True),
        evaluation.get("host_org_correct", True),
        evaluation.get("date_correct", True),
        evaluation.get("location_correct", True),
        evaluation.get("topics_correct", True),
    ]
    evaluation["all_fields_correct"] = all(fields_correct)
    evaluation["fields_accuracy"] = sum(fields_correct) / len(fields_correct)

    return evaluation


async def run_evaluation(test_cases: list[dict] = None) -> pd.DataFrame:
    """
    Run evaluation on all test cases.

    Args:
        test_cases: List of test case dictionaries. If None, uses all test cases.

    Returns:
        DataFrame with evaluation results
    """
    if test_cases is None:
        test_cases = TEST_EMAILS

    results = []

    print(f"Running evaluation on {len(test_cases)} test cases...")
    print("=" * 60)

    for i, test_case in enumerate(test_cases, 1):
        email_id = test_case["email_id"]
        subject = test_case["subject"][:50]  # Truncate for display

        print(f"\n[{i}/{len(test_cases)}] {email_id}: {subject}...")

        try:
            # Create SafeDocument and run extraction
            safe_doc = create_safe_document_from_test(test_case)
            result = await extract_invitation(safe_doc)

            # Evaluate result
            evaluation = evaluate_single_extraction(test_case, result)
            results.append(evaluation)

            # Print result
            if evaluation["classification_correct"]:
                print(f"  ✓ Classification correct: {result.__class__.__name__}")
                if evaluation.get("all_fields_correct"):
                    print("  ✓ All fields correct")
                elif "fields_accuracy" in evaluation:
                    print(f"  ⚠ Field accuracy: {evaluation['fields_accuracy']:.1%}")
            else:
                print(
                    f"  ✗ Classification WRONG: Expected {test_case['is_invitation']}, got {evaluation['actual_is_invitation']}"
                )

        except Exception as e:
            print(f"  ✗ ERROR: {str(e)}")
            results.append(
                {
                    "email_id": email_id,
                    "subject": test_case["subject"],
                    "error": str(e),
                    "classification_correct": False,
                }
            )

    print("\n" + "=" * 60)
    print("Evaluation complete!")

    return pd.DataFrame(results)

## Calculation Metrics

In [6]:
def calculate_classification_metrics(eval_df: pd.DataFrame) -> dict[str, float]:
    """Calculate classification metrics (precision, recall, F1, accuracy)."""

    # Classification metrics
    tp = len(eval_df[(eval_df["expected_is_invitation"] == True) & (eval_df["actual_is_invitation"] == True)])

    fp = len(eval_df[(eval_df["expected_is_invitation"] == False) & (eval_df["actual_is_invitation"] == True)])

    fn = len(eval_df[(eval_df["expected_is_invitation"] == True) & (eval_df["actual_is_invitation"] == False)])

    tn = len(eval_df[(eval_df["expected_is_invitation"] == False) & (eval_df["actual_is_invitation"] == False)])

    accuracy = (tp + tn) / len(eval_df) if len(eval_df) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
    }


def calculate_field_metrics(eval_df: pd.DataFrame) -> dict[str, Any]:
    """Calculate metrics for individual field extraction (for invitations only)."""

    # Only look at correctly classified invitations
    invitation_df = eval_df[
        (eval_df["expected_is_invitation"] == True) & (eval_df["actual_is_invitation"] == True)
    ].copy()

    if len(invitation_df) == 0:
        return {"note": "No correctly classified invitations to evaluate", "n": 0}

    metrics = {
        "n": len(invitation_df),
        "event_type_accuracy": invitation_df["event_type_correct"].mean(),
        "host_org_accuracy": invitation_df["host_org_correct"].mean(),
        "date_accuracy": invitation_df["date_correct"].mean(),
        "location_accuracy": invitation_df["location_correct"].mean(),
        "topics_match_rate": invitation_df["topics_correct"].mean(),  # Key fixed here
        "topics_avg_precision": invitation_df["topics_precision"].mean(),
        "topics_avg_recall": invitation_df["topics_recall"].mean(),
        "all_fields_correct_rate": invitation_df["all_fields_correct"].mean(),
        "avg_fields_accuracy": invitation_df["fields_accuracy"].mean(),
    }

    return metrics


def calculate_non_invitation_metrics(eval_df: pd.DataFrame) -> dict[str, Any]:
    """Calculate performance metrics specifically for non-invitations."""

    non_inv_df = eval_df[eval_df["expected_is_invitation"] == False].copy()

    if len(non_inv_df) == 0:
        return {"note": "No non-invitations in test set", "n": 0}

    correctly_identified = len(non_inv_df[non_inv_df["actual_is_invitation"] == False])
    incorrectly_classified = len(non_inv_df[non_inv_df["actual_is_invitation"] == True])

    accuracy = correctly_identified / len(non_inv_df)

    return {
        "n": len(non_inv_df),
        "correctly_identified": correctly_identified,
        "incorrectly_classified": incorrectly_classified,
        "accuracy": accuracy,
    }


def generate_summary_report(eval_df: pd.DataFrame) -> str:
    """Generate a human-readable summary report focused on accuracy."""

    classification_metrics = calculate_classification_metrics(eval_df)
    field_metrics = calculate_field_metrics(eval_df)
    non_inv_metrics = calculate_non_invitation_metrics(eval_df)

    report = []
    report.append("=" * 70)
    report.append("INVITATION EXTRACTION EVALUATION SUMMARY")
    report.append("=" * 70)
    report.append("")

    # Overall stats
    report.append(f"Total test cases: {len(eval_df)}")
    report.append(f"Expected invitations: {eval_df['expected_is_invitation'].sum()}")
    report.append(f"Expected non-invitations: {(~eval_df['expected_is_invitation']).sum()}")
    report.append("")

    # MAIN METRIC: Overall Accuracy
    report.append("OVERALL CLASSIFICATION ACCURACY")
    report.append("-" * 70)
    report.append(f"Overall Accuracy: {classification_metrics['accuracy']:.1%}")
    report.append(
        f"  ✓ Correct: {classification_metrics['true_positives'] + classification_metrics['true_negatives']}/{len(eval_df)}"
    )
    report.append(
        f"  ✗ Incorrect: {classification_metrics['false_positives'] + classification_metrics['false_negatives']}/{len(eval_df)}"
    )
    report.append("")

    # Breakdown by category
    report.append("PERFORMANCE BREAKDOWN")
    report.append("-" * 70)

    # Invitations
    total_invitations = eval_df["expected_is_invitation"].sum()
    correctly_classified_invitations = classification_metrics["true_positives"]
    invitation_accuracy = correctly_classified_invitations / total_invitations if total_invitations > 0 else 0

    report.append(f"Invitations ({total_invitations} cases):")
    report.append(
        f"  Accuracy: {invitation_accuracy:.1%} "
        f"({correctly_classified_invitations}/{total_invitations} correctly classified)"
    )
    report.append(f"  Missed (false negatives): {classification_metrics['false_negatives']}")
    report.append("")

    # Non-invitations
    report.append(f"Non-invitations ({non_inv_metrics['n']} cases):")
    report.append(
        f"  Accuracy: {non_inv_metrics['accuracy']:.1%} "
        f"({non_inv_metrics['correctly_identified']}/{non_inv_metrics['n']} correctly classified)"
    )
    report.append(f"  Misclassified as invitations (false positives): {non_inv_metrics['incorrectly_classified']}")
    report.append("")

    # Field extraction metrics
    if field_metrics.get("n", 0) > 0:
        report.append("FIELD EXTRACTION ACCURACY (Fuzzy/Threshold Match)")
        report.append("-" * 70)
        report.append(f"Event Type:    {field_metrics['event_type_accuracy']:.1%}")
        report.append(f"Host Org:      {field_metrics['host_org_accuracy']:.1%}")
        report.append(f"Date:          {field_metrics['date_accuracy']:.1%}")
        report.append(f"Location:      {field_metrics['location_accuracy']:.1%}")
        report.append(f"Topics (50%+): {field_metrics['topics_match_rate']:.1%}")
        report.append("")
        report.append(
            f"All Fields Correct: {field_metrics['all_fields_correct_rate']:.1%} "
            f"({int(field_metrics['all_fields_correct_rate'] * field_metrics['n'])}/{field_metrics['n']} invitations)"
        )

    report.append("")
    report.append("=" * 70)

    return "\n".join(report)


def analyse_errors(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame of all errors for detailed analysis."""

    errors = eval_df[eval_df["classification_correct"] == False].copy()

    if len(errors) == 0:
        print("No classification errors found! 🎉")
        return pd.DataFrame()

    return errors[
        [
            "email_id",
            "subject",
            "expected_is_invitation",
            "actual_is_invitation",
            "error_type",
        ]
    ]


def analyse_field_errors(eval_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Analyse field-level errors for correctly classified invitations.

    This only returns cases where the fuzzy/threshold match failed.
    """

    invitation_df = eval_df[
        (eval_df["expected_is_invitation"] == True) & (eval_df["actual_is_invitation"] == True)
    ].copy()

    if len(invitation_df) == 0:
        return {}

    field_errors = {}

    # Event type errors (Fuzzy failure)
    event_type_errors = invitation_df[invitation_df["event_type_correct"] == False]
    if len(event_type_errors) > 0:
        field_errors["event_type"] = event_type_errors[
            ["email_id", "subject", "expected_event_type", "actual_event_type"]
        ]

    # Host org errors (Fuzzy failure)
    host_errors = invitation_df[invitation_df["host_org_correct"] == False]
    if len(host_errors) > 0:
        field_errors["host_org"] = host_errors[["email_id", "subject", "expected_host_org", "actual_host_org"]]

    # Date errors (Exact match failure after normalisation)
    date_errors = invitation_df[invitation_df["date_correct"] == False]
    if len(date_errors) > 0:
        field_errors["date"] = date_errors[["email_id", "subject", "expected_date", "actual_date"]]

    # Location errors (Fuzzy failure)
    location_errors = invitation_df[invitation_df["location_correct"] == False]
    if len(location_errors) > 0:
        field_errors["location"] = location_errors[["email_id", "subject", "expected_location", "actual_location"]]

    # Topic errors (Failed the 50% recall threshold)
    topic_errors = invitation_df[invitation_df["topics_correct"] == False]
    if len(topic_errors) > 0:
        field_errors["topics"] = topic_errors[
            [
                "email_id",
                "subject",
                "expected_topics",
                "actual_topics",
                "topics_recall",
                "topics_missing",
            ]
        ]

    return field_errors

## Evaluation Output

In [4]:
async def evaluate():
    print("Starting evaluation...")
    print()
    eval_df = await run_evaluation()
    print("\n")
    summary = generate_summary_report(eval_df)
    print(summary)

    print("\n")
    print("=" * 70)
    print("ERROR ANALYSIS")
    print("=" * 70)
    classification_errors = analyse_errors(eval_df)
    if len(classification_errors) > 0:
        print("\nCLASSIFICATION ERRORS:")
        display(classification_errors)

    field_errors = analyse_field_errors(eval_df)
    if field_errors:
        print("\nFIELD EXTRACTION ERRORS:")
        for field_name, error_df in field_errors.items():
            print(f"\n{field_name.upper()} ERRORS ({len(error_df)}):")
            display(error_df)

    print("\n")
    print("=" * 70)
    print("NON-INVITATION ANALYSIS")
    print("=" * 70)

    non_invitations = eval_df[eval_df["expected_is_invitation"] == False]

    print(f"\nTotal non-invitations tested: {len(non_invitations)}")
    print(f"Correctly identified: {len(non_invitations[non_invitations['actual_is_invitation'] == False])}")
    print(
        f"Incorrectly classified as invitations: {len(non_invitations[non_invitations['actual_is_invitation'] == True])}"
    )

In [7]:
eval_df = await evaluate()

Starting evaluation...

Running evaluation on 30 test cases...

[1/30] test_001: Invitation: Tech Policy Summit 2026...
  ✓ Classification correct: Invitation
  ⚠ Field accuracy: 80.0%

[2/30] test_002: Speaking Opportunity - Climate Action Week...
  ✓ Classification correct: Invitation
  ⚠ Field accuracy: 80.0%

[3/30] test_003: Save the Date: Healthcare Innovation Forum...
  ✓ Classification correct: Invitation
  ⚠ Field accuracy: 60.0%

[4/30] test_004: Invitation to Visit Manufacturing Plant...
  ✓ Classification correct: Invitation
  ✓ All fields correct

[5/30] test_005: Education Roundtable - Late March...
  ✓ Classification correct: Invitation
  ⚠ Field accuracy: 80.0%

[6/30] test_006: Defence Briefing - 18th February...
  ✓ Classification correct: Invitation
  ✓ All fields correct

[7/30] test_007: Join us next Tuesday...
  ✓ Classification correct: Invitation
  ⚠ Field accuracy: 80.0%

[8/30] test_008: Monthly Policy Update - January 2026...
  ✓ Classification correct: NotIn

,email_id,subject,expected_event_type,actual_event_type
0,test_001,Invitation: Tech Policy Summit 2026,conference,speech
2,test_003,Save the Date: Healthcare Innovation Forum,conference,panel
18,test_019,Multi-Day Trade Mission to Japan,site_visit,other
19,test_020,Invitation: Annual Charity Gala + Policy Discu...,reception,other
27,test_028,Invitation: Virtual Webinar on Digital Transfo...,meeting,other



DATE ERRORS (4):


,email_id,subject,expected_date,actual_date
18,test_019,Multi-Day Trade Mission to Japan,2026-04-15,2020-04-15
19,test_020,Invitation: Annual Charity Gala + Policy Discu...,2026-03-28,NaN
24,test_025,🎤 SPEAKING OPPORTUNITY 🎤 Climate Summit,2026-03-20,NaN
27,test_028,Invitation: Virtual Webinar on Digital Transfo...,2026-02-27,NaN



LOCATION ERRORS (8):


,email_id,subject,expected_location,actual_location
2,test_003,Save the Date: Healthcare Innovation Forum,NaN,To be confirmed
4,test_005,Education Roundtable - Late March,NaN,Not specified
6,test_007,Join us next Tuesday,NaN,the usual place
12,test_013,Coffee Chat?,NaN,Not specified
13,test_014,Invitation: Emergency Session on Energy Crisis,NaN,Call (virtual/phone)
14,test_015,Speaking Request,NaN,Not specified
15,test_016,Reception Invitation,NaN,TBC
27,test_028,Invitation: Virtual Webinar on Digital Transfo...,Virtual/Zoom,Virtual via Zoom



TOPICS ERRORS (2):


,email_id,subject,expected_topics,actual_topics,topics_recall,topics_missing
1,test_002,Speaking Opportunity - Climate Action Week,"[climate action, government commitments, envir...","[climate change, environmental policy, net zero]",0.333333,"[climate action, government commitments]"
24,test_025,🎤 SPEAKING OPPORTUNITY 🎤 Climate Summit,"[climate commitments, climate action]","[climate change, environmental policy, net zero]",0.000000,"[climate action, climate commitments]"




NON-INVITATION ANALYSIS

Total non-invitations tested: 14
Correctly identified: 14
Incorrectly classified as invitations: 0
